# 03. Differential Expression Analysis

**목적**: Cell type 간 또는 조건 간 차등 발현 유전자(DEG) 탐색

**입력**: `output/checkpoints/{dataset}/08_annotated.h5ad`  
**출력**: `output/tables/de_results/{dataset}_*.csv`, `output/figures/de/{dataset}/`


In [ ]:
import sys
sys.path.insert(0, "../../")
sys.path.insert(0, "../")

import scanpy as sc
import pandas as pd
import numpy as np
from pathlib import Path

from modules.io import load_adata, save_adata
from modules.plotting import volcano_plot, umap_panel

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white")

In [ ]:
DATASET = "human_genes_only"
CHECKPOINT_DIR = f"../../output/checkpoints/{DATASET}"
DE_OUT_DIR = Path(f"../../output/tables/de_results/{DATASET}")
FIG_OUT_DIR = Path(f"../../output/figures/de/{DATASET}")
DE_OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_OUT_DIR.mkdir(parents=True, exist_ok=True)

adata = load_adata(f"{CHECKPOINT_DIR}/08_annotated.h5ad")
print(adata)

## 1. Cell type 간 DEG (one-vs-rest)

In [ ]:
# Wilcoxon rank-sum test (one-vs-rest)
# adata.raw (원본 count)을 사용
sc.tl.rank_genes_groups(
    adata,
    groupby="cell_type",
    method="wilcoxon",
    use_raw=True,
    key_added="rank_genes_cell_type",
    pts=True,  # 각 그룹에서 발현되는 세포 비율 계산
)
print("DEG analysis complete")

In [ ]:
# 상위 마커 유전자 dotplot
sc.pl.rank_genes_groups_dotplot(
    adata,
    key="rank_genes_cell_type",
    n_genes=5,
    groupby="cell_type",
)

In [ ]:
# 결과를 DataFrame으로 변환 후 저장
de_results = sc.get.rank_genes_groups_df(
    adata,
    group=None,  # 모든 그룹
    key="rank_genes_cell_type",
    pval_cutoff=0.05,
    log2fc_min=0.5,
)
out_file = DE_OUT_DIR / "celltype_one_vs_rest.csv"
de_results.to_csv(out_file, index=False)
print(f"Saved: {out_file}")
de_results.head(10)

## 2. Volcano plot (특정 cell type)

In [ ]:
# ============================================================
# 설정: volcano plot을 그릴 cell type
# ============================================================
TARGET_CELL_TYPE = list(adata.obs["cell_type"].unique())[0]  # 첫 번째 cell type

target_de = de_results[de_results["group"] == TARGET_CELL_TYPE]
if len(target_de) > 0:
    volcano_plot(
        target_de,
        logfc_col="logfoldchanges",
        pval_col="pvals_adj",
        gene_col="names",
        save=str(FIG_OUT_DIR / f"volcano_{TARGET_CELL_TYPE.replace(' ', '_')}.png"),
    )

## 3. 조건 간 DEG (샘플 조건 설정 후 활성화)

`samples.yaml`의 `condition`이 채워진 후 아래 코드를 사용하세요.

In [ ]:
# 조건이 알려진 경우 아래 주석 해제
# CONDITION_KEY = "condition"  # obs 칼럼명
# 
# sc.tl.rank_genes_groups(
#     adata,
#     groupby=CONDITION_KEY,
#     method="wilcoxon",
#     use_raw=True,
#     key_added="rank_genes_condition",
# )
# sc.pl.rank_genes_groups(adata, key="rank_genes_condition", n_genes=20)
print("Condition-based DEG: uncomment when condition metadata is available")

In [ ]:
# 저장 (DE 결과 포함)
save_adata(adata, f"{CHECKPOINT_DIR}/08_annotated.h5ad")
print("Saved with DE results")